# Aula 07 - Notebook: Validade de Argumentos e Inferência Lógica na Segurança de Processos

Neste notebook implementamos a classe **`ProvadorDedutivoFormal`** para testar rigorosamente a validade lógica de argumentos de segurança operacional do complexo de fertilizantes. Exploramos métodos de prova exaustiva por tabela-verdade, verificação por refutação (*Reductio ad Absurdum*) e detecção de falácias formais em lógicas de intertravamento de segurança.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

import itertools
from typing import List, Dict, Callable, Any, Tuple, Set

class ProvadorDedutivoFormal:
    @staticmethod
    def verificar_argumento_tabela_verdade(
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """
        Verifica a validade do argumento: P1, P2, ..., Pk |- C
        Um argumento e valido sse em toda linha onde todas as premissas sao TRUE,
        a conclusao tambem e estritamente TRUE.
        """
        n = len(variaveis)
        total_estados = 2 ** n
        linhas_criticas = 0 # Linhas onde todas as premissas sao verdadeiras
        linhas_validas = 0   # Linhas criticas onde a conclusao tambem e verdadeira
        contraexemplos = []
        
        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            # Avalia conjuncao de premissas
            premissas_satisfeitas = all(p(env) for p in premissas)
            
            if premissas_satisfeitas:
                linhas_criticas += 1
                if conclusao(env):
                    linhas_validas += 1
                else:
                    contraexemplos.append(env)
                    
        valido = (linhas_criticas > 0) and (linhas_criticas == linhas_validas)
        
        return {
            "Total Estados (2^n)": total_estados,
            "Estados com Premissas True": linhas_criticas,
            "Estados com Conclusão True": linhas_validas,
            "Válido": valido,
            "Resultado Semântico": "ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA)" if valido else "FALÁCIA / ARGUMENTO INVÁLIDO",
            "Contraexemplos": contraexemplos
        }

    @staticmethod
    def verificar_por_refutacao(
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """
        Prova por Contradição / Refutação (SAT Solver approach):
        O argumento P1..Pk |- C e valido se e somente se o conjunto {P1, ..., Pk, NOT C}
        for INSATISFATÍVEL (CONTRADIÇÃO).
        """
        n = len(variaveis)
        modelos_refutacao = []
        
        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            if all(p(env) for p in premissas) and not conclusao(env):
                modelos_refutacao.append(env)
                
        is_insatisfativel = len(modelos_refutacao) == 0
        return {
            "Satisfaz Negação": len(modelos_refutacao) > 0,
            "Refutação Bem-Sucedida": is_insatisfativel,
            "Conclusão": "PROVA POR CONTRADIÇÃO: ARGUMENTO VÁLIDO" if is_insatisfativel else "REFUTAÇÃO FALHOU: CONTRADIÇÃO NÃO ENCONTRADA"
        }

print("[OK] Módulo ProvadorDedutivoFormal inicializado com sucesso!")


[OK] Módulo ProvadorDedutivoFormal inicializado com sucesso!


In [2]:
# ==============================================================================
# BATERIA DE TESTES INDUSTRIAIS DE REGRAS DE INFERÊNCIA
# ==============================================================================

# 1. Modus Ponens: (p1 -> trip), p1 |- trip
vars_mp = ['p1', 'trip']
p1_mp = lambda env: (not env['p1']) or env['trip']  # Regra: p1 -> trip
p2_mp = lambda env: env['p1']                       # Fato: p1 e verdadeiro
c_mp  = lambda env: env['trip']                     # Conclusao: trip

res_mp = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_mp, [p1_mp, p2_mp], c_mp)
ref_mp = ProvadorDedutivoFormal.verificar_por_refutacao(vars_mp, [p1_mp, p2_mp], c_mp)

# 2. Modus Tollens: (bomba_on -> fluxo_ok), not fluxo_ok |- not bomba_on
vars_mt = ['bomba_on', 'fluxo_ok']
p1_mt = lambda env: (not env['bomba_on']) or env['fluxo_ok'] # bomba_on -> fluxo_ok
p2_mt = lambda env: not env['fluxo_ok']                      # not fluxo_ok
c_mt  = lambda env: not env['bomba_on']                      # not bomba_on

res_mt = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_mt, [p1_mt, p2_mt], c_mt)

# 3. Silogismo Hipotético: (vazamento -> fecha_valvula), (fecha_valvula -> isola_setor) |- (vazamento -> isola_setor)
vars_sh = ['vazamento', 'fecha_valvula', 'isola_setor']
p1_sh = lambda env: (not env['vazamento']) or env['fecha_valvula']
p2_sh = lambda env: (not env['fecha_valvula']) or env['isola_setor']
c_sh  = lambda env: (not env['vazamento']) or env['isola_setor']

res_sh = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_sh, [p1_sh, p2_sh], c_sh)

# 4. Resolução Proposicional: (sobrepressao ou falha_eletrica), (not sobrepressao ou trip_imediato) |- (falha_eletrica ou trip_imediato)
vars_res = ['sobrepressao', 'falha_eletrica', 'trip_imediato']
p1_res = lambda env: env['sobrepressao'] or env['falha_eletrica']
p2_res = lambda env: (not env['sobrepressao']) or env['trip_imediato']
c_res  = lambda env: env['falha_eletrica'] or env['trip_imediato']

res_res = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_res, [p1_res, p2_res], c_res)

# 5. Falácia da Afirmação do Consequente (INVÁLIDO): (p1 -> trip), trip |- p1
vars_fal = ['p1', 'trip']
p1_fal = lambda env: (not env['p1']) or env['trip']
p2_fal = lambda env: env['trip']
c_fal  = lambda env: env['p1']

res_fal = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_fal, [p1_fal, p2_fal], c_fal)

relatorio_testes = [
    {"Esquema Lógico": "Modus Ponens (MP)", "Variáveis": "p1, trip", "Resultado Semântico": res_mp["Resultado Semântico"], "Válido": res_mp["Válido"]},
    {"Esquema Lógico": "Modus Tollens (MT)", "Variáveis": "bomba_on, fluxo_ok", "Resultado Semântico": res_mt["Resultado Semântico"], "Válido": res_mt["Válido"]},
    {"Esquema Lógico": "Silogismo Hipotético (SH)", "Variáveis": "vazamento, fecha_valv, isola", "Resultado Semântico": res_sh["Resultado Semântico"], "Válido": res_sh["Válido"]},
    {"Esquema Lógico": "Resolução Proposicional (RES)", "Variáveis": "sobrep, falha_el, trip", "Resultado Semântico": res_res["Resultado Semântico"], "Válido": res_res["Válido"]},
    {"Esquema Lógico": "Afirmação Consequente (Falácia)", "Variáveis": "p1, trip", "Resultado Semântico": res_fal["Resultado Semântico"], "Válido": res_fal["Válido"]}
]

print("=== RELATÓRIO DE VERIFICAÇÃO FORMAL DE ARGUMENTOS DE SEGURANÇA ===")
print(formatar_tabela(relatorio_testes))

assert res_mp["Válido"] is True
assert res_mt["Válido"] is True
assert res_sh["Válido"] is True
assert res_res["Válido"] is True
assert res_fal["Válido"] is False
print("\n[OK] Todos os testes de inferência dedutiva e detecção de falácias passaram com 100% de sucesso!")


=== RELATÓRIO DE VERIFICAÇÃO FORMAL DE ARGUMENTOS DE SEGURANÇA ===
Esquema Lógico                  | Variáveis                    | Resultado Semântico                     | Válido
--------------------------------+------------------------------+-----------------------------------------+-------
Modus Ponens (MP)               | p1, trip                     | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Modus Tollens (MT)              | bomba_on, fluxo_ok           | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Silogismo Hipotético (SH)       | vazamento, fecha_valv, isola | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Resolução Proposicional (RES)   | sobrep, falha_el, trip       | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Afirmação Consequente (Falácia) | p1, trip                     | FALÁCIA / ARGUMENTO INVÁLIDO            | False 

[OK] Todos os testes de inferência dedutiva e detecção de falácias passaram com 100% de sucesso!
